In [2]:
import pandas as pd
import csv

In [ ]:
# 1. Leer CSV original
# =========================
df = pd.read_csv("../data/diputados_original_hcdn.csv")
# =========================

In [ ]:
# 2. Limpiar nombres de columnas
# =========================
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
)

# Ahora las columnas quedan:
# id, apellido, nombre, genero, distrito, inicio, fin,
# juramento, cese, bloque, bloque_inicio, bloque_fin


In [ ]:
# =========================
# 3. Limpiar textos y pasar todo a minúscula
# =========================

columnas_texto = df.select_dtypes(include="object").columns

for col in columnas_texto:
    df[col] = (
        df[col]
        .astype(str)
        .str.strip()
        .str.lower()
        .str.replace(",", "", regex=False)  # elimina comas dentro de los textos
    )

df = df.replace({
    "nan": pd.NA,
    "none": pd.NA,
    "nat": pd.NA,
    "": pd.NA
})

In [17]:
# 4. Convertir fechas y sacar horarios
# =========================

columnas_fecha = [
    "inicio",
    "fin",
    "juramento",
    "cese",
    "bloque_inicio",
    "bloque_fin"
]

for col in columnas_fecha:
    df[col] = pd.to_datetime(df[col], errors="coerce")



In [ ]:
# 5. Crear CSV histórico limpio
# =========================

df_historial = df.copy()

# Para guardar sin horario
for col in columnas_fecha:
    df_historial[col] = df_historial[col].dt.strftime("%Y-%m-%d")
df_historial.to_csv(
    "../data/diputados_historial_limpio.csv",
    index=False,
    quoting=csv.QUOTE_NONE,
    escapechar="\\"
)


In [ ]:
# =========================
# 6. Crear CSV sin diputados repetidos
# =========================

df_unicos = (
    df
    .groupby("id", as_index=False)
    .agg(
        nombre=("nombre", "first"),
        apellido=("apellido", "first"),
        genero=("genero", "first"),
        distritos=("distrito", lambda x: " | ".join(sorted(x.dropna().unique()))),
        cantidad_distritos=("distrito", "nunique"),
        cantidad_bloques=("bloque", "nunique"),
        fecha_inicio=("inicio", "min"),
        fecha_fin=("fin", "max")
    )
)

df_unicos = df_unicos.rename(columns={
    "id": "id_diputado"
})

df_unicos["nombre_completo"] = (
    df_unicos["nombre"].str.strip() + " " + df_unicos["apellido"].str.strip()
)

df_unicos = df_unicos[
    [
        "id_diputado",
        "nombre",
        "apellido",
        "nombre_completo",
        "genero",
        "distritos",
        "cantidad_distritos",
        "cantidad_bloques",
        "fecha_inicio",
        "fecha_fin"
    ]
]

df_unicos["fecha_inicio"] = df_unicos["fecha_inicio"].dt.strftime("%Y-%m-%d")
df_unicos["fecha_fin"] = df_unicos["fecha_fin"].dt.strftime("%Y-%m-%d")

df_unicos.to_csv(
    "../data/diputados_unicos.csv",
    index=False
)

,id_diputado,nombre,apellido,nombre_completo,genero,distritos,cantidad_distritos,cantidad_bloques,fecha_inicio,fecha_fin
0,hcdn0011,dulce,granados,dulce granados,f,buenos aires,1,1,2009-12-10,2017-12-09
1,hcdn0014,marcelo eduardo,lopez arias,marcelo eduardo lopez arias,m,salta,1,3,2007-12-10,2011-12-09
2,hcdn0019,mabel hilda,müller,mabel hilda müller,f,buenos aires,1,2,2009-12-10,2013-12-09
3,hcdn0022,miguel angel,pichetto,miguel angel pichetto,m,buenos aires,1,3,2023-12-10,2027-12-09
4,hcdn0035,mario raul,negri,mario raul negri,m,cordoba,1,1,2011-12-10,2023-12-09
